In [194]:
import pandas as pd
import os

In [195]:
dataframes = {}

pf_df = pd.read_json('./staging_data/raw/payload_flights.json')
payload_df = pd.DataFrame(pf_df['results'].tolist())

sf_df = pd.read_json('./staging_data/raw/spacecraft_flights.json')
spacecraft_df = pd.DataFrame(sf_df['results'].tolist())

print("payload cols", payload_df.columns)
print("spacecraft cols", spacecraft_df.columns)

payload cols Index(['response_mode', 'id', 'url', 'destination', 'amount', 'payload',
       'launch', 'landing'],
      dtype='str')
spacecraft cols Index(['id', 'url', 'destination', 'mission_end', 'spacecraft', 'launch',
       'landing', 'duration', 'turn_around_time', 'response_mode'],
      dtype='str')


In [196]:
def check_types(df:pd.DataFrame):
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            print(col)

print("========== Payload ==========")
check_types(payload_df)
print("========== Spacecraft ==========")
check_types(spacecraft_df)


========== Payload ==========
payload
launch
landing
========== Spacecraft ==========
spacecraft
launch
landing


In [197]:
pd.DataFrame(spacecraft_df['spacecraft'].tolist()).head()

,response_mode,id,url,name,serial_number,is_placeholder,image,in_space,time_in_space,time_docked,flights_count,mission_ends_count,status,description,spacecraft_config,fastest_turnaround
0,normal,541,https://lldev.thespacedevs.com/2.3.0/spacecraf...,Orion 004,004,False,"{'id': 1919, 'name': 'Orion approaching the Mo...",False,P0D,P0D,0,0,"{'id': 5, 'name': 'Under construction'}",Orion spacecraft used for Artemis 3.,"{'response_mode': 'normal', 'id': 4, 'url': 'h...",NaN
1,normal,290,https://lldev.thespacedevs.com/2.3.0/spacecraf...,Starliner 2,2,False,"{'id': 1020, 'name': 'Boeing Starliner in orbi...",False,P5DT23H54M13S,P4DT18H8M,1,1,"{'id': 1, 'name': 'Active'}",The second CST-100 Starliner capsule to fly.,"{'response_mode': 'normal', 'id': 9, 'url': 'h...",NaN
2,normal,540,https://lldev.thespacedevs.com/2.3.0/spacecraf...,Orion Integrity,003,False,"{'id': 1919, 'name': 'Orion approaching the Mo...",False,P9DT1H32M15S,P0D,1,1,"{'id': 1, 'name': 'Active'}",Orion spacecraft used for Artemis 2.,"{'response_mode': 'normal', 'id': 4, 'url': 'h...",NaN
3,normal,447,https://lldev.thespacedevs.com/2.3.0/spacecraf...,Dream Chaser Tenacity,NaN,False,"{'id': 1913, 'name': 'Dream Chaser Eagle on th...",False,P0D,P0D,0,0,"{'id': 1, 'name': 'Active'}",First operational Dream Chaser.,"{'response_mode': 'normal', 'id': 24, 'url': '...",NaN
4,normal,548,https://lldev.thespacedevs.com/2.3.0/spacecraf...,Gaganyaan G2,NaN,False,"{'id': 1914, 'name': '[AUTO] Gaganyaan - image...",False,P0D,P0D,0,0,"{'id': 4, 'name': 'Single Use'}",Second operational Gaganyaan capsule.,"{'response_mode': 'normal', 'id': 29, 'url': '...",NaN


In [198]:
######### move spacecraft_id to launch
spacecraft_launches = pd.DataFrame(spacecraft_df['launch'].tolist())
spacecraft_launches['spacecraft_id'] = pd.DataFrame(spacecraft_df['spacecraft'].tolist())['id']

launch_df = pd.concat(
    [
        pd.DataFrame(payload_df['launch'].tolist()),
        spacecraft_launches
    ],
    ignore_index=True
)

landing_df = pd.concat(
    [
        pd.DataFrame(payload_df['landing'].dropna().tolist()),
        pd.DataFrame(spacecraft_df['landing'].dropna().tolist())
    ],
    ignore_index=True
)

print("============ Launch",
        launch_df.columns.tolist(),
        launch_df.shape, sep="\n")
print("---")
check_types(launch_df)
print("============ Landing",
      landing_df.columns.tolist(),
        landing_df.shape, sep="\n")
print("---")
check_types(landing_df)


============ Launch
['id', 'url', 'name', 'response_mode', 'slug', 'launch_designator', 'status', 'last_updated', 'net', 'net_precision', 'window_end', 'window_start', 'image', 'infographic', 'probability', 'weather_concerns', 'failreason', 'hashtag', 'launch_service_provider', 'rocket', 'mission', 'pad', 'webcast_live', 'program', 'orbital_launch_attempt_count', 'location_launch_attempt_count', 'pad_launch_attempt_count', 'agency_launch_attempt_count', 'orbital_launch_attempt_count_year', 'location_launch_attempt_count_year', 'pad_launch_attempt_count_year', 'agency_launch_attempt_count_year', 'spacecraft_id']
(50, 33)
---
status
net_precision
image
launch_service_provider
rocket
mission
pad
============ Landing
['id', 'url', 'attempt', 'success', 'description', 'downrange_distance', 'landing_location', 'type']
(39, 8)
---
landing_location
type


In [199]:
rocket_df = pd.DataFrame(
    launch_df['rocket']
        .tolist()
)

rocket_df = pd.concat(
    [
        rocket_df.drop(columns=['configuration']),
        pd.json_normalize(rocket_df['configuration'])
    ],
    axis=1
)

rocket_df.to_csv('./staging_data/splitted/rockets.csv')
# rocket_df.head()

In [200]:
provider_df = pd.DataFrame(
    launch_df['launch_service_provider']
        .tolist()
)

provider_df.to_csv('./staging_data/splitted/providers.csv')

# provider_df.head()

mission_df = pd.DataFrame(
    launch_df['mission']
        .tolist()
).drop(['agencies','info_urls','vid_urls','orbit','image'],axis=1)

# mission_df.head()
mission_df.to_csv('./staging_data/splitted/missions.csv')

pad_df = pd.DataFrame(
    launch_df['pad']
        .tolist()
).drop(['agencies'],axis=1)

pad_df.to_csv('./staging_data/splitted/pads.csv')

launch_df.to_csv('./staging_data/splitted/launches.csv')


In [201]:
p_df = pd.DataFrame(
    payload_df['payload']
        .tolist()
).drop(['image'],axis=1)
check_types(p_df)

p_df.to_csv('./staging_data/splitted/payloads.csv')

agencies_df = pd.concat(
    [
        pd.DataFrame(p_df['manufacturer'].dropna().tolist()),
        pd.DataFrame(p_df['operator'].dropna().tolist())
    ],
    ignore_index=True
)

agencies_df.to_csv('./staging_data/splitted/agencies.csv')

programs_df = pd.DataFrame(
    payload_df['payload']
        .tolist()
).dropna()

programs_df.head()

programs_df.to_csv('./staging_data/splitted/programs.csv')

type
manufacturer
operator


In [203]:
s_df = pd.DataFrame(
    spacecraft_df['spacecraft'].tolist()
)

# s_df.head()

s_df.to_csv('./staging_data/splitted/spacecrafts.csv')